# Akkadian Translation (ByT5 + MBR)

Single fine-tuned **ByT5 model** with improved decoding.

## Idea

* Beam search — generates stable, high-probability translations
  `[A → A1, A2, A3, A4] (similar, safe)`

* Sampling — adds diverse alternative candidates
  `[A → B1, C7, D3, X9] (different, noisy)`

* MBR — picks the most reliable result by comparing all candidates
  `[choose argmax similarity across {A*, B*, C*}]`

* Lookup — returns exact translation if input was seen in training data
  `[A → known → direct answer]`

## Pipeline

```text id="r6y3zt"
Lookup → fallback to Beam (4 candidates) + Sampling (6 candidates) → 10 candidates → MBR → cleanup
```

## MBR

Select candidate most similar to others (Jaccard over words).


## STAGE 1 — Environment setup & data loading

In [1]:
# ============================================================
# STAGE 1 - DATA PREPARATION (CLEAN, NO LEAKAGE)
# ============================================================

import os, re, torch
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
base = "/kaggle/input/deep-past-initiative-machine-translation"

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
train_df = pd.read_csv(f"{base}/train.csv")
test_df  = pd.read_csv(f"{base}/test.csv")
pub_df   = pd.read_csv(f"{base}/published_texts.csv")
sent_df  = pd.read_csv(f"{base}/Sentences_Oare_FirstWord_LinNum.csv")

# Rename columns for consistency
train_df = train_df.rename(columns={
    'transliteration': 'akkadian',
    'translation': 'english'
})

print(f"✅ Train: {len(train_df):,}")

# ------------------------------------------------------------
# CLEAN FUNCTION
# ------------------------------------------------------------
def clean(text):
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

# ------------------------------------------------------------
# SOURCE A: published_texts
# ------------------------------------------------------------
src_a = pub_df[['transliteration', 'AICC_translation']].copy()

src_a = src_a.rename(columns={
    'transliteration': 'akkadian',
    'AICC_translation': 'english'
}).dropna()

src_a = src_a[
    (src_a['akkadian'].str.strip() != '') &
    (src_a['english'].str.strip() != '')
]

print(f"✅ Source A (published_texts): {len(src_a):,}")

# ------------------------------------------------------------
# SOURCE B: Sentences_Oare
# ------------------------------------------------------------
sent_extra = sent_df[['text_uuid', 'translation']].dropna(subset=['translation'])

pub_uuid = pub_df[['oare_id', 'transliteration']].rename(columns={
    'oare_id': 'text_uuid',
    'transliteration': 'akkadian'
})

src_b = sent_extra.merge(pub_uuid, on='text_uuid', how='left')

src_b = src_b.rename(columns={
    'translation': 'english'
})[['akkadian', 'english']].dropna()

src_b = src_b[
    (src_b['akkadian'].str.strip() != '') &
    (src_b['english'].str.strip() != '')
]

print(f"✅ Source B (Sentences_Oare): {len(src_b):,}")

# ------------------------------------------------------------
# COMBINE DATA (NO LEAKAGE)
# ------------------------------------------------------------
all_data = pd.concat([
    train_df[['akkadian', 'english']],
    src_a[['akkadian', 'english']],
    src_b[['akkadian', 'english']],
], ignore_index=True)

# ------------------------------------------------------------
# CLEAN DATA
# ------------------------------------------------------------
all_data['akkadian'] = all_data['akkadian'].apply(clean)
all_data['english']  = all_data['english'].apply(clean)

all_data = all_data.dropna()

# Length filtering
all_data = all_data[
    (all_data['akkadian'] != '') &
    (all_data['english']  != '') &
    (all_data['akkadian'].str.split().str.len() >= 2) &
    (all_data['english'].str.split().str.len()  >= 2) &
    (all_data['akkadian'].str.split().str.len() <= 300) &
    (all_data['english'].str.split().str.len()  <= 300)
]

# Remove duplicates
all_data = all_data.drop_duplicates(subset=['akkadian']).reset_index(drop=True)

print(f"\n✅ TOTAL TRAINING PAIRS: {len(all_data):,}")

✅ Train: 1,561
✅ Source A (published_texts): 7,702
✅ Source B (Sentences_Oare): 8,474

✅ TOTAL TRAINING PAIRS: 2,877


## STAGE 2 — Single model setup

In [2]:
# ------------------------------------------------------------
# SINGLE MODEL SETUP (NO SOUP)
# Using best public fine-tuned model (~34 BLEU)
# ------------------------------------------------------------
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRANSLATOR_PREFIX = "translate Akkadian to English: "

MAX_IN = 256
MAX_OUT = 192

MODEL_PATH = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"

print("Loading single optimized model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True
).to(DEVICE)

model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

print(f"SINGLE MODEL READY ✓ on {DEVICE}")

Loading single optimized model...


2026-03-18 13:38:12.026579: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773841092.204119      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773841092.265120      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773841092.695005      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773841092.695052      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773841092.695062      23 computation_placer.cc:177] computation placer alr

SINGLE MODEL READY ✓ on cuda


## STAGE 3 — Translation inference (generate predictions)

In [3]:
import numpy as np

def simple_score(a, b):
    sa = set(a.lower().split())
    sb = set(b.lower().split())
    if not sa or not sb:
        return 0
    return len(sa & sb) / len(sa | sb)

def mbr_select(cands):
    if len(cands) == 1:
        return cands[0]
    scores = []
    for i, a in enumerate(cands):
        s = 0
        for j, b in enumerate(cands):
            if i == j:
                continue
            s += simple_score(a, b)
        scores.append(s)
    return cands[np.argmax(scores)]

In [4]:
# ============================================================
# STAGE 3 - MBR INFERENCE + LOOKUP
# ============================================================

# ------------------------------------------------------------
# BUILD LOOKUP BY text_id
# ------------------------------------------------------------
from tqdm.auto import tqdm
import gc

pub_df["oare_id"] = pub_df["oare_id"].astype(str)

lookup_by_id = {
    str(row["oare_id"]): str(row["AICC_translation"]).strip()
    for _, row in pub_df.iterrows()
    if pd.notna(row["AICC_translation"])
}

def lookup_by_text_id(text_id):
    text_id = str(text_id)
    for k, v in lookup_by_id.items():
        if k.startswith(text_id):
            return v
    return None


# ------------------------------------------------------------
# MBR FUNCTIONS
# ------------------------------------------------------------

def simple_score(a, b):
    sa = set(a.lower().split())
    sb = set(b.lower().split())
    if not sa or not sb:
        return 0
    return len(sa & sb) / len(sa | sb)

def mbr_select(cands):
    if len(cands) == 1:
        return cands[0]

    scores = []
    for i, a in enumerate(cands):
        s = 0
        for j, b in enumerate(cands):
            if i != j:
                s += simple_score(a, b)
        scores.append(s)

    return cands[np.argmax(scores)]


# ------------------------------------------------------------
# INFERENCE
# ------------------------------------------------------------

model.eval()

preds = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):

    # 1. LOOKUP FIRST
    lookup = lookup_by_text_id(row["text_id"])

    if lookup is not None:
        preds.append(lookup)
        continue

    # 2. MODEL INPUT
    text = TRANSLATOR_PREFIX + clean(row["transliteration"])

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_IN
    ).to(DEVICE)

    # ----- BEAM SEARCH -----
    beam_out = model.generate(
        **enc,
        num_beams=8,
        num_return_sequences=4,
        length_penalty=1.3,
        repetition_penalty=1.2,
        max_new_tokens=MAX_OUT
    )

    beam_texts = tokenizer.batch_decode(beam_out, skip_special_tokens=True)

    # ----- SAMPLING -----
    sample_texts = []

    for temp in [0.6, 0.8, 1.05]:
        sample_out = model.generate(
            **enc,
            do_sample=True,
            temperature=temp,
            top_p=0.92,
            num_return_sequences=2,
            max_new_tokens=MAX_OUT
        )

        sample_texts.extend(
            tokenizer.batch_decode(sample_out, skip_special_tokens=True)
        )

    # ----- MBR -----
    pool = beam_texts + sample_texts
    best = mbr_select(pool)

    preds.append(best.strip())


# ------------------------------------------------------------
# CLEANUP (AFTER INFERENCE)
# ------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

  0%|          | 0/4 [00:00<?, ?it/s]

## STAGE 4 — Submission assembly

In [5]:
# ============================================================
# STAGE 4 - SUBMISSION
# ============================================================

submission = pd.DataFrame({
    'id': test_df['id'],
    'translation': preds
})

submission.to_csv('submission.csv', index=False)

print("✅ submission.csv saved")
print(submission.head())

✅ submission.csv saved
   id                                        translation
0   0  Thus says the Kanesh colony: Speak to the paym...
1   1  In the tablet of the City, you wrote to me in ...
2   2  Just as you hear our letter, there he has give...
3   3  I sent our tablet to every single place and th...


In [6]:
preview = pd.DataFrame({
    "id": test_df["id"],
    "original_transliteration": test_df["transliteration"],
    "predicted_translation": preds
})

for _, r in preview.head(10).iterrows():
    print("="*90)
    print("id:", r["id"])
    print("ORIGINAL:", r["original_transliteration"])
    print("PREDICT :", r["predicted_translation"])

id: 0
ORIGINAL: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim(ki) i-li-kam
PREDICT : Thus says the Kanesh colony: Speak to the payment of our messengers, every single single single single single single single single single single single single single single single single singl
id: 1
ORIGINAL: i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú-lá e-WA ia-ra-tí-au kà-ru-um kà-ni-ia i-lá-qé
PREDICT : In the tablet of the City, you wrote to me in the tablet of the City. On this day, whoever acquires a lapis lazuli, will not discuss in the presence of Daur, and the colony of Kanesh will take
id: 2
ORIGINAL: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-ma a-dí-ni lá i-dí-in ma-lá KÙ.AN na-áa-ú ni-bi„-it a-aí-im au-um-au ú au-mì a-bi„-au i-na mup-pì-im lu-up-ta-nim-ma ia-tí aí-ip-ri-ni aé-bi„-lá-n